# imports
here we are importing libraries

In [2]:
import pandas as pd
import yaml
import numpy as np

# write requirements.txt file from imports
with open("requirements.txt", "w") as f:
    f.write("pandas\n")
    f.write("pyyaml\n")
    f.write("numpy\n")

# just for this notebook
import pprint as pp

# Constants

In [ ]:
data_file = "oly_data/Results_Test_Version.csv"
sport = "Biathlon"
#data_file = f"data/Results_{sport}.csv"
criteria_file = f"qualification_rules/{sport}_qualification_rules.yaml"


# Read data from a CSV file using Pandas


In [4]:
def normalize_filtered_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Dates to datetime
    if "Date" in out.columns:
        out["Date_dt"] = pd.to_datetime(out["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    else:
        out["Date_dt"] = pd.NaT

    # Year to int (coerce invalid to NaN then drop in comparisons)
    if "Year" in out.columns:
        out["Year_int"] = pd.to_numeric(out["Year"], errors="coerce").astype("Int64")
    else:
        out["Year_int"] = pd.Series([pd.NA] * len(out), dtype="Int64")

    # Rank to numeric (strip non-digits like "DNS", "DNF", etc.)
    if "Rank" in out.columns:
        out["Rank_num"] = pd.to_numeric(out["Rank"], errors="coerce")
    else:
        out["Rank_num"] = np.nan

    # Normalize strings we match exactly
    for col in ["Comp.SetDetail", "CompetitionSet", "Sport", "Discipline", "Combined_Sport"]:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip()

    # Person column (entity key)
    if "Person" not in out.columns:
        out["Person"] = ""

    # create a numeric result column (try common score fields)
    out["Result_num"] = pd.to_numeric(out.get("Result", np.nan), errors="coerce")
    if out["Result_num"].isna().all():
        for alt in ["Score", "Points", "Total", "TotalScore", "Total_Score"]:
            if alt in out.columns:
                out["Result_num"] = out["Result_num"].fillna(pd.to_numeric(out[alt], errors="coerce"))

    return out

def read_data(data_file, sport="Biathlon", filter_col="Sport"):
    """
    Read a CSV file into a pandas DataFrame.
    Processes it
    filters by sport in the specified column,
    
    Args:
        data_file (str): Path to the CSV file
        sport (str): Sport to filter by
        filter_col (str): Column name to apply the sport filter on
        
    Returns:
        pd.DataFrame: DataFrame containing the processed and filtered CSV data
    """
    data = pd.read_csv(data_file, sep=";")
    data = data.drop(columns=["Phase", "YoB", "Medal"])
    data["Combined_Sport"] = data["Sport"].astype(str) + "_" + data["Discipline"]
    data = data[data[filter_col].astype(str).str.strip() == str(sport).strip()]
    if data.empty:
        return []
    else: 
        data = normalize_filtered_df(data)
    return data

In [5]:
data = read_data(data_file, sport="Biathlon", filter_col="Sport")
# display(data.head())

In [ ]:
# Display the first 5 rows of the DataFrame
#display(data.head())

# Display the column names and their data types
#display(data.info())

# Display a summary of the DataFrame
#display(data.describe())

# Display the entire DataFrame
#display(data)

# Read criteria from a YAML file using PyYAML



In [128]:
# make sure all Sports work, even if the yaml files have different formats
def normalize_criteria(rules: dict, group: str | None = "Group_A") -> dict:
    """
    Always return {"any_of": [...]} no matter how the YAML is structured.
    Supports:
      - {"selection_paths": {"any_of": [...]}}
      - {"qualification_groups": {"Group_A": {"priority": 1, "selection_paths": {...}}, ...}}
      - {"any_of": [...]}
    """
    # 1) Direct selection_paths
    if "selection_paths" in rules:
        sp = rules["selection_paths"]
        return sp if isinstance(sp, dict) and "any_of" in sp else {"any_of": sp}

    # 2) Grouped rules
    if "qualification_groups" in rules:
        groups = rules["qualification_groups"] or {}
        # explicit group if provided
        if group and group in groups:
            sp = groups[group].get("selection_paths", {})
            return sp if "any_of" in sp else {"any_of": sp}
        # else pick lowest priority
        if groups:
            best_name = min(groups, key=lambda name: groups[name].get("priority", float("inf")))
            sp = groups[best_name].get("selection_paths", {})
            return sp if "any_of" in sp else {"any_of": sp}

    # 3) Raw any_of at top-level
    if "any_of" in rules:
        return {"any_of": rules["any_of"]}

    return {"any_of": []}
# define a function to load criteria from a YAML file

def load_criteria(criteria_file):
    with open(criteria_file, "r", encoding="utf-8-sig") as criteria_file_handle:       
        text = criteria_file_handle.read().replace('\t', '  ')
        criteria = yaml.safe_load(text)
    criteria = normalize_criteria(criteria)
    return criteria

In [129]:
criteria = load_criteria(criteria_file)

In [130]:
pp.pprint(criteria)

{'any_of': [{'all_of': [{'condition': {'Comp.SetDetail': 'IBU World '
                                                         'Championships',
                                       'Rank': {'interval': [1, 3]},
                                       'Year': 2025,
                                       'count_at_least': 1,
                                       'description': 'Top-3 at World '
                                                      'Championships 2025'}},
                        {'condition': {'Comp.SetDetail': 'BMW IBU World Cup',
                                       'Date': {'interval': ['2025-11-01',
                                                             '2026-01-18']},
                                       'Rank': {'interval': [1, 30]},
                                       'count_at_least': 1,
                                       'description': 'Top-30 in World Cup '
                                                      '2025/2026'}}]},
            {'al

# Apply eligibility checker

In [131]:
def mask_for_condition(df: pd.DataFrame, cond: dict) -> pd.Series:
    """
    cond looks like:
      {
        "Comp.SetDetail": "BMW IBU World Cup",
        "Year": 2025,
        "Date": {"interval": ["2025-11-01", "2026-01-18"]},
        "Rank": {"interval": [1, 30]},
        "count_at_least": 1
      }
    """
    #initialize all as True
    mask = pd.Series(True, index=df.index)

    # Check if Competition "BMW IBU World Cup" matches
    if "Comp.SetDetail" in cond:
        mask &= (df.get("Comp.SetDetail", "") == str(cond["Comp.SetDetail"]).strip())

    #Check if the year of the competition matches
    if "Year" in cond:
        # Compare to normalized Int64 year
        target_year = cond["Year"]
        mask &= (df["Year_int"] == target_year)

    # Interval checks (inclusive)
    if "Date" in cond and isinstance(cond["Date"], dict) and "interval" in cond["Date"]:
        start_str, end_str = cond["Date"]["interval"]
        start = pd.to_datetime(start_str, utc=True).tz_localize(None)
        end = pd.to_datetime(end_str, utc=True).tz_localize(None)
        mask &= (df["Date_dt"] >= start) & (df["Date_dt"] <= end)

    if "Rank" in cond and isinstance(cond["Rank"], dict) and "interval" in cond["Rank"]:
        lo, hi = cond["Rank"]["interval"]
        mask &= (df["Rank_num"] >= lo) & (df["Rank_num"] <= hi)

    # case-insensitive equality for Discipline / Gender, added this part to make Figure Skating work
    if "Discipline" in cond:
        left = df.get("Discipline", "").astype(str).str.strip().str.casefold()
        right = str(cond["Discipline"]).strip().casefold()
        mask &= (left == right)

    if "Gender" in cond:
        left = df.get("Gender", "").astype(str).str.strip().str.casefold()
        right = str(cond["Gender"]).strip().casefold()
        mask &= (left == right)

    # NEW: numeric Result thresholds
    if "Result" in cond and isinstance(cond["Result"], dict):
        r = cond["Result"]
        val = pd.to_numeric(df.get("Result_num", np.nan), errors="coerce")
        if "interval" in r:
            lo, hi = map(float, r["interval"])
            mask &= (val >= lo) & (val <= hi)
        if "ge" in r:
            mask &= (val >= float(r["ge"]))
        if "le" in r:
            mask &= (val <= float(r["le"]))
    #print(f"mask_for_condition: {mask.sum()} rows match condition {cond}")
    return mask


def eligible_for_condition(df: pd.DataFrame, mask: pd.Series, count_at_least: int, entity_col: str = "Person") -> set:
    """
    Given a mask, return set of entities that appear at least count_at_least times.
    """
    hits = df.loc[mask, entity_col].dropna()
    if hits.empty:
        return set()
    counts = hits.value_counts()
    #print(f"eligible_for_condition: found {len(counts[counts >= count_at_least])} entities with at least {count_at_least} hits.")       
    return set(counts[counts >= count_at_least].index)

def eval_all_of(df: pd.DataFrame, path: dict, entity_col: str = "Person") -> set:
    """
    path can be:
      - {"all_of": [{"condition": {...}}, {"condition": {...}}]}
      - {"condition": {...}}   # treat as single-step all_of
    """
    # Normalize: if a bare "condition" is provided, treat it as all_of with one step
    if "condition" in path and "all_of" not in path:
        path = {"all_of": [ {"condition": path["condition"]} ]}

    groups = []
    for step in path.get("all_of", []):
        cond = step.get("condition", {})
        mask = mask_for_condition(df, cond)
        threshold = int(cond.get("count_at_least", 1))
        ok = eligible_for_condition(df, mask, threshold, entity_col=entity_col)
        groups.append(ok)

    if not groups:
        return set()

    eligible = groups[0]
    for s in groups[1:]:
        eligible &= s
    #print(f"eval_all_of: found {len(eligible)} eligible entities across {len(groups)} groups.")
    return eligible

def eval_any_of(df: pd.DataFrame, rules: dict, entity_col: str = "Person") -> set:
    """
    rules: {"any_of": [ {"condition": {...}}, {"all_of": [...]}, ... ]}
    Returns union of entities that satisfy at least one path.
    """
    print("Entering eval_any_of")
    winners = set()
    #print(f"eval_any_of: processing {len(rules.get('any_of', []))} paths.")
    for path in rules.get("any_of", []):
        # Delegate to eval_all_of which now handles both shapes
        winners |= eval_all_of(df, path, entity_col=entity_col)
        #print(f"eval_any_of: path resulted in {len(winners)} winners so far.")
    #print(f"eval_any_of: found {len(winners)} winners across {len(rules.get('any_of', []))} paths.")
    return winners

def eligibility_checker(
    rules: dict,
    df: pd.DataFrame,        # set to "Combined_Sport" if that's what you're passing
) -> list[str]:

    eligible_set = eval_any_of(df, rules, entity_col="Person")
    #print(f"Found {len(eligible_set)} eligible athletes.")

    return sorted(eligible_set)

In [132]:

eligible_athletes = eligibility_checker(criteria, data)
print(eligible_athletes)

Entering eval_any_of
['Aita Gasparin', 'Amy Baserga', 'Elisa Gasparin', 'Jeremy Finello', 'Joscha Burkhalter', 'Lena Häcki-Groß', 'Niklas Hartweg', 'Sebastian Stalder']
